В этом ноутбуке мы будем работать над моделью предсказания движеняи цены акций по новостям, чтобы выявить важность признаков, а также актуальность бизнес-задачи.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv("finhub+theguardian_raw_7.5k.csv", index_col=0)
df = df.dropna(subset=["fields.trailText"])
df.shape

В нашем датасете есть новости для которых есть краткая сводка, а есть новости для которых есть и краткая сводка и полный текст статьи. Для удобства будет обозначать их как short и full (или же s и f)

In [ ]:
df["fields.trailText"]

In [ ]:
df["fields.bodyText"]

Ячейками ниже преобразуем наши текста в эмбеддинги. Код ниже был запущен в ноутубке на kaggle, поскольку на CPU данная операция работает очень медленно. Код был перенесен сюда и закомментирован, чтобы он не выполнялся при нажатии на "Run All".

In [ ]:
# from sentence_transformers import SentenceTransformer

# model = SentenceTransformer("intfloat/multilingual-e5-small")

# texts = ["passage: " + x for x in df["fields.bodyText"]]

# embeddings_full = model.encode(
#     texts,
#     normalize_embeddings=True,
#     show_progress_bar=True
# )

In [ ]:
# model = SentenceTransformer("intfloat/multilingual-e5-small")

# texts = ["passage: " + x for x in df["fields.trailText"]]

# embeddings_short = model.encode(
#     texts,
#     normalize_embeddings=True,
#     show_progress_bar=True
# )


Сохраняем наши эмбеддинги в файлы

In [ ]:
# np.save("embeddings_f.npy", embeddings_full)

In [ ]:
# np.save("embeddings_s.npy", embeddings_short)

Сразу же подгрузим и преобразуем в DataFrame

In [ ]:
np_embeddings_full = np.load('embeddings_f.npy')
np_embeddings_short = np.load('embeddings_s.npy')

In [ ]:
embeddings_full = pd.DataFrame(np_embeddings_full)
embeddings_short = pd.DataFrame(np_embeddings_short)

In [ ]:
embeddings_full.shape

In [ ]:

embeddings_short.shape

Соединим эмбеддинги с исходным датасетом, из которого оставим только то что пригодится для нашей модели - дата публикации, тикеры которые были указаны экспертами в статье, компании для которых эта новость была найдена, и источник новости.

In [ ]:
embeddings_full.columns = [f"emb_f_{i}" for i in range(embeddings_full.shape[1])]
embeddings_short.columns = [f"emb_s_{i}" for i in range(embeddings_short.shape[1])]

In [ ]:
df = df[["webPublicationDate", "stocks", "company", "source"]]

In [ ]:
data = pd.concat(
    [
        df.reset_index(drop=True),
        embeddings_full.reset_index(drop=True),
        embeddings_short.reset_index(drop=True)
    ],
    axis=1
)

В столбце company для статей из FinHub есть пропуски, зато в тикерах пропусков нет. Восстановим компании по тикерам:

In [ ]:
mapping = {
    "AAPL": "Apple",
    "TSLA": "Tesla",
    "CAT": "Caterpillar",
    "JPM": "JPMorgan",
    "NFLX": "Netflix",
    "PFE": "Pfizer",
    "XOM": "Exxon",
    "WMT": "Walmart"
}

In [ ]:
data.loc[data["company"].isna(), "company"] = data.loc[data["company"].isna(), "stocks"].str.split(";").apply(lambda x: ";".join(mapping[i] for i in x if i in mapping))

In [ ]:
data.isna().sum(axis=0)

Для столбца stocks сделаем OHE оставив только нужные нам тикеры:

In [ ]:
stocks = (data["stocks"].fillna("").str.get_dummies(sep=";"))[["AAPL", "TSLA", "CAT", "JPM", "NFLX", "PFE", "XOM", "WMT"]]

In [ ]:
data = pd.concat([stocks, data], axis=1)

Аналогично сделаем OHE для столбца source:

In [ ]:
source = data["source"].str.get_dummies()

In [ ]:
data = pd.concat([source, data], axis=1)

In [ ]:
data = data.drop(columns=["stocks", "source"])

Сохраним наш датасет под названием data_model

In [ ]:
# data.to_csv("data_model.csv", index=False)

Разделим его на новости у которых есть полное содержимое и у которых нет:

In [ ]:
common_cols = [c for c in data.columns if not c.startswith("emb_f_") and not c.startswith("emb_s_")]
f_cols = [c for c in data.columns if c.startswith("emb_f_")]
s_cols = [c for c in data.columns if c.startswith("emb_s_")]

data_f = data[common_cols + f_cols].copy()
data_s = data[common_cols + s_cols].copy()

In [ ]:
data_f = data_f.dropna()
data_s = data_s.dropna()

In [ ]:
# data_f.to_csv("data_f.csv", index=False)
# data_s.to_csv("data_s.csv", index=False)

In [ ]:
data_f = pd.read_csv("data_f.csv")
data_s = pd.read_csv("data_s.csv")

In [ ]:
data_f["webPublicationDate"] = pd.to_datetime(data_f["webPublicationDate"])
data_s["webPublicationDate"] = pd.to_datetime(data_s["webPublicationDate"])

В столбце company у нас несколько компаний для одной и той же новости. Рзаделим их так, чтобы одна строка с несколькими компаниями превратилась в несколько разных строк, каждая с одной компанией. Так будет легче выделять нужные нам строки:

In [ ]:
data_f["company"] = data_f["company"].str.split(";")
data_f = data_f.explode("company", ignore_index=True)

In [ ]:
data_s["company"] = data_s["company"].str.split(";")
data_s = data_s.explode("company", ignore_index=True)

In [ ]:
data_f.head()

In [ ]:
data_f["webPublicationDate"]

Нам нужно вывести таргет для нашей модели. У нас в качестве таргета будет являться логарифм от процентного изменения цен. Изменение цен будем считать относительно последней цены, на которую текущая новость не повлияла - назовем такую цену P0. P1 - это цена закрытия в следующую торговую сессию. Pn - цена закрытия на n-ный торговую сессию после P0. А таргет соотвественно - log(Pn/P0) который мы будем считать для разных n с целью выявить лучший.

Чтобы получить последнюю цену закрытия, на которую текущая новость не повлияла, надо добавить само время закрытия в датасет котировок. В США это 20:00 для летнего времени и 21:00 для обычного:

In [ ]:
quotes = pd.read_csv("quotes_all.csv", index_col=0)

In [ ]:
quotes["Date"] = pd.to_datetime(quotes["Date"], utc=True)

In [ ]:
quotes["is_summer"] = quotes["Date"].apply(lambda x: (x.month > 3 or (x.month == 3 and x.day >= 8)) and (x.month < 11 or (x.month == 11 and x.day < 1)))

In [ ]:
quotes["Date"] = quotes["Date"] + pd.to_timedelta(np.where(quotes["is_summer"], "20:00:00", "21:00:00"))

In [ ]:
quotes.head()

Приведем тикеры к названиям компаний

In [ ]:
quotes["company"] = quotes["ticker"].map(mapping)

In [ ]:
quotes.head()

Отсортируем все по дате

In [ ]:
quotes = quotes.sort_values(["Date", "ticker"]).reset_index(drop=True)
data_f = data_f.sort_values(["webPublicationDate", "company"]).reset_index(drop=True)

Теперь создадим колонку с номером котировки для каждого тикера:

In [ ]:
quotes["q_idx"] = quotes.groupby("ticker").cumcount()

In [ ]:
quotes.head()

Сначала найдем P0 - это самая поздняя цена закрытия на которую не повлияла новость. С этим нам поможет pd.merge_asof который делает мердж по ближайшему значению ключа. 

In [ ]:
data_f = pd.merge_asof(
    data_f,
    quotes,
    left_on="webPublicationDate",
    right_on="Date",
    by="company",
    direction="backward"
)

In [ ]:
data_f.head()

Собственно Adj Close в самом конце это и есть наша P0. Как получить P1, P2 и следующие? Поскольку датафрейм уже отсортирован в рамках тикера, то мы пройтись по ним и построить матрицу будущих цен:

In [ ]:
for k in range(31):
    quotes[f"P{k}"] = quotes.groupby("ticker")["Adj Close"].shift(-k)

In [ ]:
quotes.head()

Осталось только присоединить эти значения к исходному датасету:

In [ ]:
data_f = data_f.merge(
    quotes[["company", "q_idx"] + [f"P{k}" for k in range(31)]],
    on=["company", "q_idx"],
    how="left"
)

In [ ]:
data_f.head()

In [ ]:
data_f = data_f.dropna(axis=0)

Отлично, данные готовы, можем делить их на X и y:

In [ ]:
y = data_f[["company"] + [f"P{k}" for k in range(31)]]
X = data_f.drop(columns=["ticker", "webPublicationDate", "Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "is_summer", "q_idx"] + [f"P{k}" for k in range(31)])

In [ ]:
y.head()

Но не забудем что в y мы работаем именно с измнением цены:

In [ ]:
y[[f"P{i}" for i in range(1, 31)]] = np.log(y[[f"P{i}" for i in range(1, 31)]].div(y["P0"], axis=0))

In [ ]:
y.head()

Попробуем поработать с компанией Apple.

In [ ]:
X_apple = X[X["company"] == "Apple"].drop(columns=["company"])
y_apple = y[y["company"] == "Apple"].drop(columns=["company", "P0"])

In [ ]:
y_apple_bin = (y_apple[[f"P{i}" for i in range(1, 31)]] > 0).astype(int)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

for i in range(1, 31):
    X_apple_train, X_apple_test, y_apple_train, y_apple_test = train_test_split(X_apple, y_apple_bin[f"P{i}"], test_size=0.2, random_state=67)

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_apple_train, y_apple_train)

    pred = model.predict(X_apple_test)
    prob = model.predict_proba(X_apple_test)[:, 1]

    print(f"F1 для P{i}: {f1_score(y_apple_test, pred)}")
    print(f"ROC-AUC для P{i}: {roc_auc_score(y_apple_test, prob)}")
    print()

Наилучший F1 при задержке в 15 дней. Однако ROC-AUC оставляет желать лучшего - прогноз как у случайного угадывания.

Вывод: Модель в текущем виде не пригодна для практического использования. Нужно возвращаться к анализу данных и выбору алгоритма.